# Preprocessing (HPC-optimised)

Extracts crops from a video based on trajectory files.
Key I/O optimisations:

- **Async write queue**: crops are written by background threads so the main loop never blocks on disk.
- **Zero PNG compression**: fastest possible writes (`cv2.IMWRITE_PNG_COMPRESSION, 0`).
- **Pre-built skip-set**: existing crop filenames are collected once with `glob` instead of one `exists()` syscall per frame.
- **Numpy crop buffer**: the white canvas is kept in CPU memory; only the current video frame is transferred GPU → CPU once per frame.


In [11]:
import os
from pathlib import Path
import threading
import queue
import cv2
import torch
import numpy as np
from tqdm import tqdm

## Device & Paths


In [12]:
# Check whether a GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEBUG: Processing on device: {device}")

data_dir = Path("/scratch/cvcdt011/data")
if not data_dir.exists():
    data_dir = Path("../data")


DEBUG: Processing on device: cuda


## Configuration


In [13]:
HALF_SIZE = 50  # Half-width/height of each crop in pixels
CROP_SIZE = 2 * HALF_SIZE

Z_THRESHOLD = 4.0  # Z-score threshold for outlier detection

NUM_WRITERS = 4  # Background writer threads
WRITE_QUEUE_MAX = 512  # Max crops buffered in memory before the main loop blocks

PNG_PARAMS = [cv2.IMWRITE_PNG_COMPRESSION, 3]  # Default compression level

# Group members must be able to read AND write the outputs
CROP_FILE_MODE = 0o664  # rw-rw-r--
CROP_DIR_MODE = 0o775  # rwxrwxr-x (x needed on dirs to enter/list them)

## Helper: Read Next Row


In [14]:
def read_next_row(fh):
    """Read and parse the next line from a trajectory file handle.
    Returns (frame_id, x, y, angle) or None if EOF.
    """
    line = fh.readline()
    if not line:
        return None
    parts = line.strip().split(",")
    return (int(parts[0]), float(parts[1]), float(parts[2]), float(parts[4]))

## Outlier Detection (Z-score for Position and Orientation)


In [ ]:
def detect_trajectory_outliers(file_path, z_threshold=Z_THRESHOLD):
    """Reads a trajectory file, computes displacement and angular change between consecutive steps,
    and returns a set of frame IDs flagged as outliers based on Z-scores.
    """
    rows = []
    with open(file_path, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) >= 5:
                rows.append(
                    (int(parts[0]), float(parts[1]), float(parts[2]), float(parts[4]))
                )

    if len(rows) < 3:
        return set()

    frames = [r[0] for r in rows]
    xs = np.array([r[1] for r in rows])
    ys = np.array([r[2] for r in rows])
    thetas = np.array([r[3] for r in rows])

    # Position jumps (displacement) and orientation changes between frames
    dx = np.diff(xs)
    dy = np.diff(ys)
    displacements = np.sqrt(dx**2 + dy**2)

    d_theta = np.diff(thetas)
    d_theta = (d_theta + 180) % 360 - 180
    abs_d_theta = np.abs(d_theta)

    outlier_frames = set()

    # Compute z-scores for displacements
    if len(displacements) > 1:
        mean_disp = np.mean(displacements)
        std_disp = np.std(displacements)
        if std_disp > 1e-6:
            z_disp = (displacements - mean_disp) / std_disp
            for idx in np.where(np.abs(z_disp) > z_threshold)[0]:
                outlier_frames.add(frames[idx + 1])

    # Compute z-scores for orientation changes
    if len(abs_d_theta) > 1:
        mean_theta = np.mean(abs_d_theta)
        std_theta = np.std(abs_d_theta)
        if std_theta > 1e-6:
            z_theta = (abs_d_theta - mean_theta) / std_theta
            for idx in np.where(np.abs(z_theta) > z_threshold)[0]:
                outlier_frames.add(frames[idx + 1])

    return outlier_frames

## Async Write Queue


In [16]:
write_queue = queue.Queue(maxsize=WRITE_QUEUE_MAX)
write_errors = []  # (path, repr(exception)) collected across writer threads


def _writer_worker():
    """Background thread: pulls (path, ndarray, angle) from the queue and writes to disk."""
    from PIL import Image, PngImagePlugin
    import cv2
    while True:
        item = write_queue.get()
        if item is None:  # Sentinel: shut down
            write_queue.task_done()
            break
        
        if len(item) == 3:
            path, img, angle = item
        else:
            path, img = item
            angle = None
            
        try:
            # OpenCV keeps images in BGR format, convert to RGB for PIL
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_pil = Image.fromarray(img_rgb)
            metadata = PngImagePlugin.PngInfo()
            if angle is not None:
                metadata.add_text("OrientationAngle", str(angle))
            # Save using PIL with zero compression to maintain fast HPC I/O
            img_pil.save(str(path), "PNG", pnginfo=metadata, compress_level=0)
            os.chmod(path, CROP_FILE_MODE)  # Group-writable regardless of umask
        except Exception as e:
            write_errors.append((path, repr(e)))  # list.append is thread-safe
            print(f"DEBUG: Write error {path}: {e}")
        write_queue.task_done()


writer_threads = []
for _ in range(NUM_WRITERS):
    t = threading.Thread(target=_writer_worker, daemon=True)
    t.start()
    writer_threads.append(t)

print(f"DEBUG: Started {NUM_WRITERS} writer threads.")

DEBUG: Started 4 writer threads.


## Preprocessing Setup


In [17]:
video_path = data_dir / "rec1.mp4"
trajectory_dir = data_dir / "rec1_trajectories"
if not trajectory_dir.exists():
    trajectory_dir = Path("data/rec1_trajectories")
crops_dir = data_dir / "crops"

# Fail fast on missing inputs — cv2.VideoCapture would otherwise fail silently later.
if not video_path.exists():
    raise FileNotFoundError(f"Video not found: {video_path}")
if not trajectory_dir.is_dir():
    raise FileNotFoundError(f"Trajectory directory not found: {trajectory_dir}")

txt_files = sorted(list(trajectory_dir.glob("*.txt")))
if not txt_files:
    raise RuntimeError(f"No trajectory .txt files in {trajectory_dir}")


def make_group_writable_dir(path):
    """mkdir -p, then force group rwx (mkdir's mode is masked by the umask,
    and only the owner may chmod a dir created by a groupmate)."""
    path.mkdir(parents=True, exist_ok=True)
    try:
        os.chmod(path, CROP_DIR_MODE)
    except PermissionError:
        pass  # Dir owned by another group member — they already set its perms


file_handles = []

print(f"DEBUG: Found {len(txt_files)} trajectory files.")

make_group_writable_dir(crops_dir)

# One directory per bee/trajectory: crops/<traj_id>/frame_XXXXXX.png
for f in txt_files:
    fh = open(f, "r")
    file_handles.append((f.stem, fh))
    make_group_writable_dir(crops_dir / f.stem)

DEBUG: Found 362 trajectory files.


## Precompute Outliers & Existing Crops


In [ ]:
# --- Outliers ---
outliers_dict = {}
total_outliers_flagged = 0
for f in txt_files:
    outliers = detect_trajectory_outliers(f)
    outliers_dict[f.stem] = outliers
    total_outliers_flagged += len(outliers)
print(
    f"DEBUG: Flagged {total_outliers_flagged} outlier frames across {len(txt_files)} trajectories."
)

# --- Existing crops: one directory listing per trajectory instead of per-frame exists() calls ---
# filename format: crops/<traj_id>/frame_000123.png
existing_crops = {}
for f in txt_files:
    tid = f.stem
    existing = set()
    crop_dir = crops_dir / tid
    for p in crop_dir.iterdir():
        try:
            frame_num = int(p.stem.split("_")[1])
        except (IndexError, ValueError):
            continue
        if frame_num in outliers_dict[tid]:
            p.unlink()  # Remove leftover outlier crops from previous runs
        else:
            existing.add(frame_num)
    existing_crops[tid] = existing

print("DEBUG: Existing-crops index built.")

In [ ]:
# Pre-load first row of each trajectory
next_rows = {}
for tid, fh in file_handles:
    row = read_next_row(fh)
    if row:
        next_rows[tid] = row

## Main Crop Extraction Loop


In [ ]:
# Pre-allocate white canvas in CPU numpy memory (reused across all crops)
white_canvas = np.full((CROP_SIZE, CROP_SIZE, 3), 255, dtype=np.uint8)

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(
        f"OpenCV could not open {video_path} — the file exists but is unreadable "
        "from this node (missing ffmpeg/codec support in this OpenCV build?)."
    )
cap.set(cv2.CAP_PROP_BUFFERSIZE, 8)  # Increase video decoder prefetch buffer
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if total_frames <= 0:
    raise RuntimeError(
        f"OpenCV reports {total_frames} frames for {video_path} — decoder cannot read it."
    )
print(f"DEBUG: Video opened, {total_frames} frames.")

frame_count = 0
stats = {"skipped": 0, "processed": 0, "outliers_filtered": 0}
pbar = tqdm(total=total_frames, desc="Extracting crops")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w = frame.shape[:2]

        for traj_id, fh in file_handles:
            if traj_id not in next_rows or next_rows[traj_id][0] != frame_count:
                continue

            # Outlier: skip and do not save crop
            if frame_count in outliers_dict.get(traj_id, set()):
                stats["outliers_filtered"] += 1

            # Already on disk: skip
            elif frame_count in existing_crops.get(traj_id, set()):
                stats["skipped"] += 1

            # New crop: extract and enqueue for async write
            else:
                try:
                    x, y = (
                        int(round(next_rows[traj_id][1])),
                        int(round(next_rows[traj_id][2])),
                    )

                    # Desired window bounds
                    y_start = x - HALF_SIZE
                    y_end = x + HALF_SIZE
                    x_start = y - HALF_SIZE
                    x_end = y + HALF_SIZE

                    # Clamped bounds (intersection with frame)
                    ymin_f = max(0, y_start)
                    ymax_f = min(h, y_end)
                    xmin_f = max(0, x_start)
                    xmax_f = min(w, x_end)

                    # Copy white canvas and paste valid region
                    crop = white_canvas.copy()
                    crop[
                        ymin_f - y_start : ymax_f - y_start,
                        xmin_f - x_start : xmax_f - x_start,
                    ] = frame[ymin_f:ymax_f, xmin_f:xmax_f]

                    angle = float(next_rows[traj_id][3])

                    crop_filename = crops_dir / traj_id / f"frame_{frame_count:06d}.png"
                    write_queue.put((crop_filename, crop, angle))  # Non-blocking enqueue
                    stats["processed"] += 1
                except Exception as e:
                    print(
                        f"DEBUG: Error processing {traj_id} at frame {frame_count}: {e}"
                    )

            row = read_next_row(fh)
            if row:
                next_rows[traj_id] = row
            else:
                next_rows.pop(traj_id, None)

        pbar.update(1)
        if frame_count % 250 == 0:
            pbar.set_postfix(
                {
                    "processed": stats["processed"],
                    "skipped": stats["skipped"],
                    "outliers": stats["outliers_filtered"],
                }
            )
        frame_count += 1

finally:
    pbar.close()
    cap.release()
    for _, fh in file_handles:
        fh.close()

# Wait for all pending writes to finish, then shut down worker threads
write_queue.join()
for _ in writer_threads:
    write_queue.put(None)
for t in writer_threads:
    t.join()

print(
    f"DEBUG: Finished. Read {frame_count}/{total_frames} frames. "
    f"Processed: {stats['processed']}, Skipped: {stats['skipped']}, "
    f"Outliers filtered: {stats['outliers_filtered']}, Write errors: {len(write_errors)}"
)

# Fail loudly instead of ending a run that produced nothing / lost crops.
if frame_count == 0:
    raise RuntimeError(f"Read 0 frames from {video_path} — decoder failed on the first frame.")
if stats["processed"] == 0 and stats["skipped"] == 0:
    raise RuntimeError(
        "No crops written and none already on disk — trajectory frame IDs "
        f"matched none of the {frame_count} video frames read."
    )
if write_errors:
    for path, err in write_errors[:10]:
        print(f"DEBUG: write error: {path}: {err}")
    raise RuntimeError(
        f"{len(write_errors)} of {stats['processed']} crop writes failed "
        f"(first: {write_errors[0][1]})."
    )

## Outlier Visualization
Visualizes a trajectory and highlights the filtered outliers in red.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_outliers(traj_id):
    txt_file = trajectory_dir / f"{traj_id}.txt"
    rows = []
    with open(txt_file, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) >= 5:
                rows.append((int(parts[0]), float(parts[1]), float(parts[2]), float(parts[4])))
    
    if len(rows) < 3:
        return
        
    frames = np.array([r[0] for r in rows])
    xs = np.array([r[1] for r in rows])
    ys = np.array([r[2] for r in rows])
    thetas = np.array([r[3] for r in rows])
    
    dx = np.diff(xs)
    dy = np.diff(ys)
    displacements = np.sqrt(dx**2 + dy**2)
    
    d_theta = np.diff(thetas)
    d_theta = (d_theta + 180) % 360 - 180
    abs_d_theta = np.abs(d_theta)
    
    pos_outliers = set()
    ang_outliers = set()
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    
    # --- Position Analysis ---
    if len(displacements) > 1:
        mean_disp = np.mean(displacements)
        std_disp = np.std(displacements)
        if std_disp > 1e-6:
            z_disp = (displacements - mean_disp) / std_disp
            for idx in np.where(np.abs(z_disp) > Z_THRESHOLD)[0]:
                pos_outliers.add(frames[idx + 1])
            
            # Plot Threshold
            thresh_disp = mean_disp + Z_THRESHOLD * std_disp
            ax2.axhline(thresh_disp, color='red', linestyle='--', label=f'Threshold ({Z_THRESHOLD}σ)')
                
    # --- Angular Analysis ---
    if len(abs_d_theta) > 1:
        mean_theta = np.mean(abs_d_theta)
        std_theta = np.std(abs_d_theta)
        if std_theta > 1e-6:
            z_theta = (abs_d_theta - mean_theta) / std_theta
            for idx in np.where(np.abs(z_theta) > Z_THRESHOLD)[0]:
                ang_outliers.add(frames[idx + 1])
            
            # Plot Threshold
            thresh_theta = mean_theta + Z_THRESHOLD * std_theta
            ax3.axhline(thresh_theta, color='red', linestyle='--', label=f'Threshold ({Z_THRESHOLD}σ)')

    # --- Plotting Trajectory ---
    ax1.plot(xs, ys, label='Trajectory', color='blue', alpha=0.5, marker='.', zorder=1)
    
    # Scatter points for outliers
    pos_x = [xs[i] for i, f in enumerate(frames) if f in pos_outliers and f not in ang_outliers]
    pos_y = [ys[i] for i, f in enumerate(frames) if f in pos_outliers and f not in ang_outliers]
    if pos_x:
        ax1.scatter(pos_x, pos_y, color='red', label='Positional Outlier', zorder=5, s=50)
        
    ang_x = [xs[i] for i, f in enumerate(frames) if f in ang_outliers and f not in pos_outliers]
    ang_y = [ys[i] for i, f in enumerate(frames) if f in ang_outliers and f not in pos_outliers]
    if ang_x:
        ax1.scatter(ang_x, ang_y, color='orange', label='Angular Outlier', zorder=4, s=50)
        
    both_x = [xs[i] for i, f in enumerate(frames) if f in pos_outliers and f in ang_outliers]
    both_y = [ys[i] for i, f in enumerate(frames) if f in pos_outliers and f in ang_outliers]
    if both_x:
        ax1.scatter(both_x, both_y, color='purple', label='Both', zorder=6, s=70, marker='*')
    
    ax1.set_title(f'Trajectory {traj_id}')
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.legend()
    ax1.invert_yaxis()
    
    # --- Boxplots ---
    ax2.boxplot(displacements, vert=True)
    ax2.set_title('Displacements (px)')
    ax2.set_ylabel('Distance')
    ax2.set_xticks([1])
    ax2.set_xticklabels(['Delta Pos'])
    ax2.legend()
    
    ax3.boxplot(abs_d_theta, vert=True)
    ax3.set_title('Angular Changes (deg)')
    ax3.set_ylabel('Degrees')
    ax3.set_xticks([1])
    ax3.set_xticklabels(['Delta Theta'])
    ax3.legend()
    
    plt.tight_layout()
    plt.show()

# Execution
traj_with_outliers = [tid for tid, outs in outliers_dict.items() if len(outs) > 0]
if traj_with_outliers:
    print(f"Found {len(traj_with_outliers)} trajectories with outliers.")
    plot_outliers(traj_with_outliers[0])
else:
    print("No outliers found to visualize.")

## Interactive Outlier Crop Viewer
Allows interactive inspection of outlier transitions by loading the corresponding frames from the video and showing the crops side-by-side along with the local trajectory context.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def extract_crop_from_frame(frame, x, y, half_size=HALF_SIZE):
    h, w = frame.shape[:2]
    # Desired window bounds (matching preprocessing crop logic)
    y_start = int(round(x)) - half_size
    y_end = int(round(x)) + half_size
    x_start = int(round(y)) - half_size
    x_end = int(round(y)) + half_size

    # Clamped bounds (intersection with frame)
    ymin_f = max(0, y_start)
    ymax_f = min(h, y_end)
    xmin_f = max(0, x_start)
    xmax_f = min(w, x_end)

    # Copy white canvas and paste valid region
    crop_size = 2 * half_size
    crop = np.full((crop_size, crop_size, 3), 255, dtype=np.uint8)
    
    c_ymin = ymin_f - y_start
    c_ymax = ymax_f - y_start
    c_xmin = xmin_f - x_start
    c_xmax = xmax_f - x_start
    
    crop[c_ymin:c_ymax, c_xmin:c_xmax] = frame[ymin_f:ymax_f, xmin_f:xmax_f]
    return crop

def get_frames_and_crops(video_path, f_prev, f_curr, coord_prev, coord_curr, half_size=HALF_SIZE):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None, None
        
    # Read prev frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, f_prev)
    ret, frame_prev = cap.read()
    if not ret:
        cap.release()
        return None, None
        
    # Read curr frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, f_curr)
    ret, frame_curr = cap.read()
    cap.release()
    if not ret:
        return None, None
        
    # Extract crops
    crop_prev = extract_crop_from_frame(frame_prev, coord_prev[0], coord_prev[1], half_size)
    crop_curr = extract_crop_from_frame(frame_curr, coord_curr[0], coord_curr[1], half_size)
    
    # Convert BGR to RGB for plotting
    crop_prev_rgb = cv2.cvtColor(crop_prev, cv2.COLOR_BGR2RGB)
    crop_curr_rgb = cv2.cvtColor(crop_curr, cv2.COLOR_BGR2RGB)
    
    return crop_prev_rgb, crop_curr_rgb

def get_outlier_transitions(traj_id, z_threshold=Z_THRESHOLD):
    txt_file = trajectory_dir / f"{traj_id}.txt"
    rows = []
    with open(txt_file, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) >= 5:
                rows.append((int(parts[0]), float(parts[1]), float(parts[2]), float(parts[4])))
    
    if len(rows) < 2:
        return []
    
    frames = [r[0] for r in rows]
    xs = np.array([r[1] for r in rows])
    ys = np.array([r[2] for r in rows])
    thetas = np.array([r[3] for r in rows])
    
    dx = np.diff(xs)
    dy = np.diff(ys)
    displacements = np.sqrt(dx**2 + dy**2)
    
    d_theta = np.diff(thetas)
    d_theta = (d_theta + 180) % 360 - 180
    abs_d_theta = np.abs(d_theta)
    
    transitions = []
    
    # Compute z-scores for displacements
    z_disp = np.zeros_like(displacements)
    if len(displacements) > 1:
        mean_disp = np.mean(displacements)
        std_disp = np.std(displacements)
        if std_disp > 1e-6:
            z_disp = (displacements - mean_disp) / std_disp
            
    # Compute z-scores for orientation changes
    z_theta = np.zeros_like(abs_d_theta)
    if len(abs_d_theta) > 1:
        mean_theta = np.mean(abs_d_theta)
        std_theta = np.std(abs_d_theta)
        if std_theta > 1e-6:
            z_theta = (abs_d_theta - mean_theta) / std_theta
            
    for idx in range(len(displacements)):
        zd = abs(z_disp[idx])
        zt = abs(z_theta[idx])
        is_outlier = (zd > z_threshold) or (zt > z_threshold)
        if is_outlier:
            transitions.append({
                "prev_idx": idx,
                "curr_idx": idx + 1,
                "prev_frame": frames[idx],
                "curr_frame": frames[idx + 1],
                "prev_coord": (xs[idx], ys[idx], thetas[idx]),
                "curr_coord": (xs[idx+1], ys[idx+1], thetas[idx+1]),
                "displacement": displacements[idx],
                "z_disp": z_disp[idx],
                "angular_change": d_theta[idx],
                "z_theta": z_theta[idx],
                "type": "Positional" if (zd > z_threshold and zt <= z_threshold) else
                        "Angular" if (zt > z_threshold and zd <= z_threshold) else "Both"
            })
            
    return transitions

def plot_outlier_details(traj_id, transition, half_size=HALF_SIZE):
    f_prev = transition["prev_frame"]
    f_curr = transition["curr_frame"]
    coord_prev = transition["prev_coord"]
    coord_curr = transition["curr_coord"]
    
    # Get crops
    crop_prev, crop_curr = get_frames_and_crops(video_path, f_prev, f_curr, coord_prev, coord_curr, half_size)
    if crop_prev is None or crop_curr is None:
        print(f"Error: Could not load frames {f_prev} and {f_curr} from {video_path}")
        return
        
    # Load local trajectory path for context
    txt_file = trajectory_dir / f"{traj_id}.txt"
    rows = []
    with open(txt_file, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) >= 5:
                rows.append((int(parts[0]), float(parts[1]), float(parts[2])))
                
    frames = [r[0] for r in rows]
    xs = [r[1] for r in rows]
    ys = [r[2] for r in rows]
    
    # Find indices around the current transition
    curr_idx = transition["curr_idx"]
    start_idx = max(0, curr_idx - 10)
    end_idx = min(len(frames), curr_idx + 11)
    
    local_frames = frames[start_idx:end_idx]
    local_xs = xs[start_idx:end_idx]
    local_ys = ys[start_idx:end_idx]
    
    # Plot
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    
    # Calculate orientation vector components (compass: 0=up, 90=right, clockwise)
    arrow_length = half_size * 0.7
    
    theta_prev = coord_prev[2]
    angle_rad_prev = np.radians(theta_prev)
    dx_prev = arrow_length * np.sin(angle_rad_prev)
    dy_prev = -arrow_length * np.cos(angle_rad_prev)
    
    theta_curr = coord_curr[2]
    angle_rad_curr = np.radians(theta_curr)
    dx_curr = arrow_length * np.sin(angle_rad_curr)
    dy_curr = -arrow_length * np.cos(angle_rad_curr)
    
    # Crop Prev
    ax1.imshow(crop_prev)
    ax1.quiver(half_size, half_size, dx_prev, dy_prev, angles='xy', scale_units='xy', scale=1, color='red', width=0.015, headwidth=4, headlength=5)
    ax1.set_title(f"Before Outlier: Frame {f_prev}\nCoord: ({coord_prev[0]:.1f}, {coord_prev[1]:.1f}), Angle: {coord_prev[2]:.1f}°")
    ax1.axis("off")
    
    # Crop Curr
    ax2.imshow(crop_curr)
    ax2.quiver(half_size, half_size, dx_curr, dy_curr, angles='xy', scale_units='xy', scale=1, color='red', width=0.015, headwidth=4, headlength=5)
    ax2.set_title(f"After Outlier: Frame {f_curr}\nCoord: ({coord_curr[0]:.1f}, {coord_curr[1]:.1f}), Angle: {coord_curr[2]:.1f}°")
    ax2.axis("off")
    
    # Trajectory Segment
    ax3.plot(local_xs, local_ys, 'b.-', label="Trajectory", alpha=0.6)
    
    # Highlight the outlier jump in red
    prev_idx_local = curr_idx - start_idx - 1
    curr_idx_local = curr_idx - start_idx
    ax3.plot(
        [local_xs[prev_idx_local], local_xs[curr_idx_local]],
        [local_ys[prev_idx_local], local_ys[curr_idx_local]],
        'r.-', linewidth=2.5, markersize=10, label=f"Outlier Jump ({transition['type']})"
    )
    
    # Add frame number labels to points
    for idx, (lx, ly, lf) in enumerate(zip(local_xs, local_ys, local_frames)):
        if lf in (f_prev, f_curr):
            ax3.annotate(f"F{lf}", (lx, ly), textcoords="offset points", xytext=(0,10), ha='center', weight='bold')
        elif idx % 2 == 0 or len(local_frames) < 8:
            ax3.annotate(f"F{lf}", (lx, ly), textcoords="offset points", xytext=(0,5), ha='center', fontsize=8, alpha=0.7)
            
    ax3.set_title(f"Trajectory Jump Detail\nDisplacement: {transition['displacement']:.2f} px (Z={transition['z_disp']:.2f})\nAngular: {transition['angular_change']:.2f}° (Z={transition['z_theta']:.2f})")
    ax3.set_xlabel("X")
    ax3.set_ylabel("Y")
    ax3.grid(True, linestyle=":", alpha=0.6)
    ax3.legend()
    ax3.invert_yaxis()
    
    plt.tight_layout()
    plt.show()

# Filter trajectories that actually have outliers
trajs_with_outliers = sorted([tid for tid, outs in outliers_dict.items() if len(outs) > 0])

if not trajs_with_outliers:
    print("No trajectories have outliers with Z_THRESHOLD =", Z_THRESHOLD)
else:
    # Dropdown for selecting Trajectory
    traj_dropdown = widgets.Dropdown(
        options=trajs_with_outliers,
        value=trajs_with_outliers[0],
        description='Trajectory:',
        style={'description_width': 'initial'}
    )
    
    # Label to show info
    info_label = widgets.HTML(value="")
    
    # Dropdown for selecting Transition index
    trans_dropdown = widgets.Dropdown(
        description='Outlier Index:',
        style={'description_width': 'initial'}
    )
    
    # Navigation buttons
    prev_button = widgets.Button(description="◀ Prev", layout=widgets.Layout(width='80px'))
    next_button = widgets.Button(description="Next ▶", layout=widgets.Layout(width='80px'))
    
    # Output area
    out = widgets.Output()
    
    # Store current transitions list
    current_transitions = []
    
    def update_transitions_dropdown(traj_id):
        global current_transitions
        current_transitions = get_outlier_transitions(traj_id)
        
        options = []
        for i, t in enumerate(current_transitions):
            options.append((f"{i+1}/{len(current_transitions)}: F{t['prev_frame']}->F{t['curr_frame']} ({t['type']})", i))
            
        trans_dropdown.options = options
        if options:
            trans_dropdown.value = 0
            info_label.value = f"<b>Total Outliers:</b> {len(current_transitions)}"
        else:
            trans_dropdown.options = [("No outliers", -1)]
            trans_dropdown.value = -1
            info_label.value = "<b>No outliers found for this trajectory.</b>"
            
    def on_traj_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            update_transitions_dropdown(change['new'])
            
    def on_trans_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            idx = change['new']
            with out:
                clear_output(wait=True)
                if idx is not None and idx >= 0 and idx < len(current_transitions):
                    plot_outlier_details(traj_dropdown.value, current_transitions[idx])
                elif idx == -1:
                    print("No transitions to plot.")
                    
    def on_prev_clicked(b):
        if trans_dropdown.value is not None and trans_dropdown.value > 0:
            trans_dropdown.value -= 1
            
    def on_next_clicked(b):
        if trans_dropdown.value is not None and trans_dropdown.value < len(current_transitions) - 1:
            trans_dropdown.value += 1
            
    traj_dropdown.observe(on_traj_change)
    trans_dropdown.observe(on_trans_change)
    prev_button.on_click(on_prev_clicked)
    next_button.on_click(on_next_clicked)
    
    # Initialize
    update_transitions_dropdown(traj_dropdown.value)
    
    # Layout UI
    controls = widgets.HBox([traj_dropdown, prev_button, trans_dropdown, next_button, info_label])
    ui = widgets.VBox([controls, out])
    display(ui)
    
    # Trigger initial plot
    with out:
        clear_output(wait=True)
        if current_transitions:
            plot_outlier_details(traj_dropdown.value, current_transitions[0])
